In [ ]:
from datetime import datetime
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator, PercentFormatter
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import zscore, linregress
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import scale, PolynomialFeatures
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
import os

In [2]:
# ---------------------
# Formatting functions
# ---------------------

def axis_formatter(dependent_variable):
    """Return a formatter function based on the dependent variable."""
    if isinstance(dependent_variable, pd.Series):
        max_value = dependent_variable.max()
    else:
        raise ValueError("dependent_variable must be a numeric pandas Series.")

    def format_ticks(x, pos):
        if "Rate" in dependent_variable.name:
            x *= 100  # Multiply by 100 to format as percentage
            if max_value < 0.00008:
                return f'{x:,.4f}%'
            elif max_value < 0.0008:
                return f'{x:,.3f}%'
            elif max_value < 0.008:
                return f'{x:,.2f}%'
            elif max_value < .08:
                return f'{x:,.1f}%' 
            else:
                return f'{int(x)}%'
        else:
            if max_value < 0.0008:
                return f'{x:,.5f}'
            elif max_value < 0.008:
                return f'{x:,.4f}'
            elif max_value < 0.08:
                return f'{x:,.3f}'
            elif max_value < 0.8:
                return f'{x:,.2f}'
            elif max_value < 8:
                return f'{x:,.1f}'
            else:
                return f'{int(x):,}'

    return format_ticks

def thousands_formatter(max_coefficient, dependent_variable):
        """Return a formatter function based on the max coefficient and the dependent variable."""
        def format_ticks(x, pos):
            # Check if the dependent variable includes 'Rate'
            if "Rate" in dependent_variable:
                x *= 100  # Multiply by 100 to format as percentage
                if max_coefficient < .00008:
                    return f'{x:,.4f}%'  # Format with five decimal places and percentage
                elif max_coefficient < .0008:
                    return f'{x:,.3f}%'  # Format with four decimal places and percentage
                elif max_coefficient < .008:
                    return f'{x:,.2f}%'  # Format with three decimal places and percentage
                elif max_coefficient < .08:
                    return f'{x:,.1f}%'  # Format with two decimal places and percentage
                else:
                    return f'{int(x)}%'  # Format without decimal place, add percentage
            else:
                # If 'Rate' is not in the dependent variable, use the normal formatting
                if max_coefficient < .0008:
                    return f'{x:,.5f}'  # Format with five decimal places
                elif max_coefficient < .008:
                    return f'{x:,.4f}'  # Format with four decimal places
                elif max_coefficient < .08:
                    return f'{x:,.3f}'  # Format with three decimal places
                elif max_coefficient < .8:
                    return f'{x:,.2f}'  # Format with two decimal places
                elif max_coefficient < 8:
                    return f'{x:,.1f}'  # Format with one decimal place
                else:
                    return f'{int(x):,}'  # Format without decimal place

        return format_ticks
    
def thousands_formatter_with_plus(max_coefficient, dependent_variable):
        """Return a formatter function based on the max coefficient and the dependent variable."""
        def format_ticks(x, pos):
            # Add a '+' for positive ticks
            sign = "+" if x > 0 else ""

            # Check if the dependent variable includes 'Rate'
            if "Rate" in dependent_variable:
                x *= 100  # Multiply by 100 to format as percentage
                if max_coefficient < .000008:
                    return f'{sign}{x:,.5f}%'  # Format with five decimal places and percentage
                elif max_coefficient < .00008:
                    return f'{sign}{x:,.4f}%'  # Format with four decimal places and percentage
                elif max_coefficient < .0008:
                    return f'{sign}{x:,.3f}%'  # Format with three decimal places and percentage
                elif max_coefficient < .008:
                    return f'{sign}{x:,.2f}%'  # Format with two decimal places and percentage
                elif max_coefficient < .08:
                    return f'{sign}{x:,.1f}%'  # Format with one decimal place and percentage
                else:
                    return f'{sign}{int(x)}%'  # Format without decimal place, add percentage
            else:
                # If 'Rate' is not in the dependent variable, use the normal formatting
                if max_coefficient < .0008:
                    return f'{sign}{x:,.5f}'  # Format with five decimal places
                elif max_coefficient < .008:
                    return f'{sign}{x:,.4f}'  # Format with four decimal places
                elif max_coefficient < .08:
                    return f'{sign}{x:,.3f}'  # Format with three decimal places
                elif max_coefficient < .8:
                    return f'{sign}{x:,.2f}'  # Format with two decimal places
                elif max_coefficient < 8:
                    return f'{sign}{x:,.1f}'  # Format with one decimal place
                else:
                    return f'{sign}{int(x):,}'  # Format without decimal place

        return format_ticks 

In [ ]:
# ======================
# Multiple regression function; iterates over each dependent variable in a list
# ======================
def multiple_regression_unstandardized(
    df, 
    target,
    dependent_variables, 
    independent_variables, 
    start_date, 
    end_date, 
    plot_residuals=False,
    variance_method="standardized",  # Options: "standardized", "semi_partial", "semi_partial_standardized"
):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import statsmodels.api as sm
    from sklearn.preprocessing import StandardScaler
    from matplotlib.ticker import FuncFormatter
    from scipy.stats import zscore
    
    # Build variance explained table columns dynamically
    variance_cols = ['Dependent Variable', 
                     'Total Variance Explained'] + \
                    [f'Variance Explained by {label}' for label in variance_categories.values()]
    
    # Initialize the variance explained table
    variance_explained_table = pd.DataFrame(columns=variance_cols)

    # Initialize the variable significance table
    variable_significance_table = pd.DataFrame(0, index=dependent_variables, columns=independent_variables)

    model_error_table = pd.DataFrame()

    # ----------------------
    # Iterate over each dependent variable and remove outliers
    # ----------------------
    for dependent_variable in dependent_variables:
        
        # Always start from the full dataset
        df_regression = df.copy()
        
        # Remove outliers in the dependent variable if it has more than X unique values
        if df[dependent_variable].nunique() > unique_value_threshold:
            
            # Count rows
            rows_before = len(df)
                     
            # Calculate the z-scores for the dependent variable
            z_scores = zscore(df[dependent_variable])

            # Identify outliers with absolute z-scores greater than x
            outliers = np.abs(z_scores) > dependent_variable_outlier_threshold

            # Remove outliers and copy to new df
            df_regression = df_regression[~outliers]

            # Count rows after removing outliers
            rows_after = len(df_regression)
            
            print(f"{rows_before - rows_after} rows removed due to outliers in '{dependent_variable}'. {rows_after} rows analyzed")

        # Reset index after removing rows
        df_regression = df_regression.reset_index(drop=True)
        
        # Extract dependent and independent variables
        y = pd.to_numeric(df_regression[dependent_variable], errors="coerce")
        x = df_regression[independent_variables].apply(pd.to_numeric, errors="coerce")

        # Replace inf values and drop incomplete rows
        regression_data = (
            pd.concat([y, x], axis=1)
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        rows_before_model = len(df_regression)
        rows_after_model = len(regression_data)
        if rows_before_model != rows_after_model:
            print(
                f"{rows_before_model - rows_after_model} rows removed due to NaN/inf in "
                f"'{dependent_variable}' or predictors. {rows_after_model} rows analyzed"
            )

        y = regression_data[dependent_variable]
        x = regression_data[independent_variables]

        # Do NOT standardize — use raw values
        x_with_const = sm.add_constant(x)

        # Fit multiple regression on unstandardized variables
        model = sm.OLS(y, x_with_const).fit()

        # Get coefficient covariance matrix, excluding intercept
        cov_matrix = model.cov_params().drop(
            index="const",
            columns="const",
            errors="ignore"
        )

        # Flatten covariance matrix into columns dynamically
        cov_values = {}

        for var1 in cov_matrix.index:
            for var2 in cov_matrix.columns:
                cov_values[f"Covariance: {var1} | {var2}"] = cov_matrix.loc[var1, var2]

        # Get Residual Std Error
        residual_std_error = model.mse_resid ** 0.5

        model_error_table = pd.concat([
            model_error_table,
            pd.DataFrame([{
                "Dependent Variable": dependent_variable,
                "Residual Std Error": residual_std_error,
                "R Squared": model.rsquared,
                "Adjusted R Squared": model.rsquared_adj,
                "Observations": int(model.nobs),
                **cov_values,
            }])
        ], ignore_index=True)

        # Print Model Summary
        print(model.summary())

        # Extract p-values and coefficients
        p_values = model.pvalues.drop(['const'])
        coefficients = model.params.drop(['const'])

        # Calculate adjusted R-squared percentage
        rsquared_adj_percentage = model.rsquared_adj * 100  # Convert to percentage
        
        # Filter for significant variables
        significant_mask = p_values < significance_level
        significant_coefficients = coefficients[significant_mask]

        # ----------------------
        # Compute variance explained per category
        # ----------------------
        if variance_method == "standardized":
            # Standardize all variables
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            x_scaled = pd.DataFrame(scaler_x.fit_transform(x), columns=independent_variables)
            y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), name=y.name)

            x_scaled_with_const = sm.add_constant(x_scaled)
            model_std = sm.OLS(y_scaled, x_scaled_with_const).fit()
            beta_coeffs = pd.Series(model_std.params[1:], index=independent_variables)

            # Sum absolute beta coefficients per category
            sums = {
                name: beta_coeffs[beta_coeffs.index.str.contains(condition)].abs().sum()
                for condition, name in variance_categories.items()
            }

        elif variance_method == "semi_partial":
            # Semi-partial R²: drop each predictor and calculate R² reduction on raw values
            r2_full = model.rsquared
            semi_partial_r2 = {}
            for predictor in independent_variables:
                x_reduced = x.drop(columns=[predictor])
                x_reduced_with_const = sm.add_constant(x_reduced)
                model_reduced = sm.OLS(y, x_reduced_with_const).fit()
                semi_partial_r2[predictor] = r2_full - model_reduced.rsquared

            # Sum semi-partial R² by category
            sums = {}
            for condition, name in variance_categories.items():
                predictors_in_cat = [pred for pred in independent_variables if condition in pred]
                sums[name] = sum([semi_partial_r2.get(pred, 0) for pred in predictors_in_cat])

        elif variance_method == "semi_partial_standardized":
            # Standardize variables first
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            x_scaled = pd.DataFrame(scaler_x.fit_transform(x), columns=independent_variables)
            y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), name=y.name)

            # Semi-partial R² on standardized variables
            r2_full_std = sm.OLS(y_scaled, sm.add_constant(x_scaled)).fit().rsquared
            semi_partial_r2_std = {}

            for predictor in independent_variables:
                x_reduced = x_scaled.drop(columns=[predictor])
                x_reduced_with_const = sm.add_constant(x_reduced)
                model_reduced = sm.OLS(y_scaled, x_reduced_with_const).fit()
                semi_partial_r2_std[predictor] = r2_full_std - model_reduced.rsquared

            # Sum semi-partial R² by category
            sums = {}
            for condition, name in variance_categories.items():
                predictors_in_cat = [pred for pred in independent_variables if condition in pred]
                sums[name] = sum([semi_partial_r2_std.get(pred, 0) for pred in predictors_in_cat])


        total_sum = sum(sums.values())

        ratios = {
            name: ((sum_val / total_sum) * rsquared_adj_percentage) if total_sum != 0 else 0
            for name, sum_val in sums.items()
        }

        total_ratio = sum(ratios.values())
        
        # Plot only selected variables
        #significant_coefficients = significant_coefficients[significant_coefficients.index.str.contains("Tagged|Highest Academe Today Position")]
        
        # Remove "Tagged " from tag variables for plotting
        significant_coefficients.index = significant_coefficients.index.str.replace("Tagged: ", "")

        # Guard against empty results
        if significant_coefficients.empty:
            plt.figure(figsize=(12, 3))
            plt.text(
                0.5, 0.5,
                "No statistically significant variables",
                ha="center", va="center", fontsize=14
            )
            plt.axis("off")
            plt.show()

        else:
            plt.figure(figsize=(12, 10))
            significant_coefficients.sort_values(ascending=False).plot(kind='bar', color=color)

            plt.title(f'Effects on {dependent_variable} per {section} {target} {start_date} to {end_date}')
            plt.ylabel(f'Change in {section} {target} {dependent_variable} for each variable')
            plt.tight_layout()

            max_coefficient = significant_coefficients.max()
            plt.gca().yaxis.set_major_formatter(
                FuncFormatter(thousands_formatter_with_plus(max_coefficient, dependent_variable))
            )

            # Add adjusted R-squared value as text
            text_lines = [f'These variables explain {rsquared_adj_percentage:.0f}% of the variance in {dependent_variable}:']
            for name in variance_categories.values():
                text_lines.append(f'{name} explains {ratios.get(name, 0):.0f}%')
            text_lines.append('Only significant variables are plotted')

            plt.text(
                x=0.995, y=0.99,
                s='\n'.join(text_lines),
                fontsize=10, color='black',
                ha='right', va='top',
                transform=plt.gca().transAxes
            )

            # Mean value annotation
            mean_value = df_regression[dependent_variable].mean()
            mean_value_formatted = thousands_formatter(mean_value, dependent_variable)(mean_value, None)

            plt.text(
                x=0.005, y=0.99,
                s=f'Mean: {mean_value_formatted} {dependent_variable} per {target}',
                fontsize=10, color='black',
                ha='left', va='top',
                transform=plt.gca().transAxes
            )

            plt.show()
        
# ----------------------
# Residuals
# ----------------------
        if plot_residuals:
            # ----------------------
            # Plot Residuals
            # ----------------------
            # Get residuals
            residuals = model.resid

            # Get the predicted values
            predicted_values = model.fittedvalues

            # Plot residuals: x-axis is observed, y-axis is residuals
            plt.figure(figsize=(12, 10))
            plt.scatter(df_regression[dependent_variable], residuals, s=5)  # Decrease dot size to 5

            plt.title(f'{section} Residuals: {dependent_variable}')
            plt.xlabel('Observed Values')
            plt.ylabel('Predicted Values')
            plt.tight_layout()
            plt.show()


            # ----------------------
            # Plot standardized residuals 
            # ----------------------
            # Compute the standardized residuals
            std_residuals = residuals / residuals.std()

            # Compute the standardized predicted values
            std_predicted_values = predicted_values / predicted_values.std()

            # Create a plot for standardized residuals v. standardized predicted values
            plt.scatter(std_predicted_values, std_residuals, s=5)
            plt.title(f'{section} Standardized Residuals: {dependent_variable}')
            plt.xlabel('Standardized Predicted Values')
            plt.ylabel('Standardized Residuals')
            plt.tight_layout()
            plt.show()

# ----------------------
# Scatter Plots
# ----------------------
        # loop through each independent variable
        for independent_variable in independent_variables:
            try:
                # Skip if the independent variable has less than 100 unique values
                if df_regression[independent_variable].nunique() < plotting_threshold:
                    continue
                    
                # Keep only valid rows for this variable pair
                filtered_df = df_regression[[independent_variable, dependent_variable]].copy()

                # Force numeric and remove bad values
                filtered_df[independent_variable] = pd.to_numeric(filtered_df[independent_variable], errors="coerce")
                filtered_df[dependent_variable] = pd.to_numeric(filtered_df[dependent_variable], errors="coerce")

                filtered_df = (
                    filtered_df
                    .replace([np.inf, -np.inf], np.nan)
                    .dropna()
                )

                # Skip if too few usable rows remain
                if len(filtered_df) < 3:
                    print(f"Skipping {independent_variable} v. {dependent_variable}: not enough valid rows after dropping NaN/inf")
                    continue

                # Set the plot size using figsize
                plt.figure(figsize=(12, 10))

                # Plot scatter plot with smaller dots, without adding it to the legend
                plt.scatter(
                    filtered_df[independent_variable],
                    filtered_df[dependent_variable],
                    s=5, alpha=0.7, label="_nolegend_"
                )

                # Fit linear regression
                slope, intercept, r_value, p_value, std_err = stats.linregress(
                    filtered_df[independent_variable],
                    filtered_df[dependent_variable]
                )
                y_pred_linear = slope * filtered_df[independent_variable] + intercept
                r2_linear = r_value ** 2  # R-squared for linear fit

                # Variables to track the best fit polynomial
                best_r2 = r2_linear
                best_degree = 1
                best_y_pred = y_pred_linear

                # Start with the linear model as the best model
                previous_r2 = r2_linear  # Initially, the linear fit is the best

                # Try polynomial fits for degrees 2 through X (or up to any desired degree)
                for degree in range(2, 6):  # start from degree 2 to try more complex fits
                    poly = PolynomialFeatures(degree=degree)
                    X_poly = poly.fit_transform(filtered_df[independent_variable].values.reshape(-1, 1))
                    poly_model = LinearRegression().fit(X_poly, filtered_df[dependent_variable])
                    y_pred_poly = poly_model.predict(X_poly)
                    r2_poly = poly_model.score(X_poly, filtered_df[dependent_variable])  # R-squared for polynomial fit

                    # Keep track of the best model based on R-squared, use only if it's X% better than the previous polynomial fit
                    if r2_poly > previous_r2 * best_fit_improvement_factor:
                        best_r2 = r2_poly
                        best_degree = degree
                        best_y_pred = y_pred_poly
                        previous_r2 = r2_poly  # Update the previous R-squared to the current polynomial's R-squared

                # Calculate the correlation coefficient
                corr_coef = filtered_df[independent_variable].corr(filtered_df[dependent_variable])

                # Plot the best fit model only if correlation is X or greater
                if abs(corr_coef) >= line_plot_threshold:
                    # Sort independent variable for smooth line plot
                    sorted_idx = np.argsort(filtered_df[independent_variable])  # Get sorted indices
                    sorted_x = filtered_df[independent_variable].iloc[sorted_idx]
                    sorted_y = best_y_pred[sorted_idx]  # Directly use the sorted indices to align predictions

                    # Plot the best fit model line with legend text showing correlation
                    plt.plot(sorted_x, sorted_y, linewidth=2,
                             label=f"Best Fit (Degree {best_degree})\nr = {corr_coef:.2f}")
                else:
                    # Add correlation to the legend even without plotting the line
                    plt.plot([], [], ' ', label=f"No Best Fit Line\nr = {corr_coef:.2f}")

                # Add titles and labels
                plt.title(f'{section} Correlation: {independent_variable} v. {target} {dependent_variable}')
                plt.xlabel(independent_variable)
                plt.ylabel(dependent_variable)
 
                plt.gca().yaxis.set_major_formatter(FuncFormatter(axis_formatter(filtered_df[dependent_variable])))
                plt.gca().xaxis.set_major_formatter(FuncFormatter(axis_formatter(filtered_df[independent_variable])))

                # Show the plot with the best fit line in the legend only
                plt.legend()
                plt.show()
                
                # Loop through the independent variables to fill the variable significance table
                for independent_variable in coefficients.index:  # Iterate through all independent variables
                    is_significant = significant_mask.get(independent_variable, False)
                    coefficient_value = coefficients.get(independent_variable, None)

                    if is_significant:  # Check if the coefficient is significant
                        variable_significance_table.at[dependent_variable, independent_variable] = 1 if coefficient_value > 0 else -1

            except Exception as e:
                print(f"Error occurred while plotting {independent_variable} v. {dependent_variable} Correlation: {e}")
                continue  # Skip to the next plot if an error occurs

# ======================
# Variance Explained Export
# ======================
        # Create a temporary DataFrame for the current results
        # Build the current result dynamically using ratios
        current_result = {'Dependent Variable': dependent_variable, 'Total Variance Explained': total_ratio / 100}
        for name in variance_categories.values():
            current_result[f'Variance Explained by {name}'] = ratios.get(name, 0) / 100

        # Convert to DataFrame and append the result with the variance_explained_table
        variance_explained_table = pd.concat([variance_explained_table, pd.DataFrame([current_result])], ignore_index=True)

    print("\nModel Error Summary:\n")
    display(model_error_table)

    # Format numeric columns as percentages without decimals
    variance_explained_table.update(
        variance_explained_table.select_dtypes(include=['number']).applymap("{:.0%}".format)
    )

    # Plot the table
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.axis("off")
    table = ax.table(cellText=variance_explained_table.values, colLabels=variance_explained_table.columns, cellLoc="center", loc="center")

    # Set font size and style (make text bigger)
    table.auto_set_font_size(True)

    # Adjust column width
    table.auto_set_column_width(col=list(range(len(variance_explained_table.columns))))

    # Adjust cell formatting
    for (i, j), cell in table.get_celld().items():
        cell.set_edgecolor('gray')
        cell.set_linewidth(0.5)

    # Left justify the first column data cells (except for header)
    for (i, j), cell in table.get_celld().items():
        if j == 0 and i != 0:
            cell.set_text_props(ha='left')

    # Save as image
    plt.savefig(f"{section} Multiple Regression Variance Explained {start_date} to {end_date}.png", bbox_inches="tight", dpi=300)
    plt.show()

    # Export the results table
    variance_explained_table.to_csv(f"{section} Multiple Regression Variance Explained {start_date} to {end_date}.csv", index=False)

# ======================
# Variable Significance Export
# ======================
    # Apply conditional formatting for Excel
    def color_cells(val):
        if val == 1:
            return 'background-color: green'
        elif val == -1:
            return 'background-color: red'
        return ''

    # Create a styled DataFrame
    styled_results = variable_significance_table.style.applymap(color_cells)

    # Save the styled table as an Excel file with conditional formatting
    styled_results.to_excel(f"{section} Multiple Regression Variable Significance {start_date} to {end_date}.xlsx", engine='openpyxl')

    # Set up figure and axis for table display
    fig, ax = plt.subplots(figsize=(10, 6))  # Adjust size for better visibility
    ax.axis("off")

    # Create a color map for the cells (excluding the index column)
    cell_colors = []
    for i in range(variable_significance_table.shape[0]):
        row_colors = []
        for j in range(variable_significance_table.shape[1]):
            val = variable_significance_table.iat[i, j]
            if val == 1:  # Positive coefficient
                row_colors.append('green')
            elif val == -1:  # Negative coefficient
                row_colors.append('red')
            else:  # No significant coefficient
                row_colors.append('white')
        cell_colors.append(row_colors)

    # Prepare the table data, moving the index to a column
    table_data = variable_significance_table.reset_index().rename(columns={'index': 'Dependent Variables'})

    # Create the table in the plot
    table = ax.table(cellText=table_data.values, 
                     colLabels=table_data.columns, 
                     cellLoc="center", 
                     loc="center")

    # Apply font and style, and color the cells
    for (i, j), cell in table.get_celld().items():
        cell.set_text_props(ha='center')  # Center text in all cells
        if i == 0:  # Header row
            cell.set_fontsize(12)
            cell.set_text_props(weight='bold')
        elif j == 0:  # First column (Dependent Variables) - no conditional formatting
            cell.set_facecolor('white')
            cell.set_fontsize(10)
        else:
            cell.set_facecolor(cell_colors[i-1][j-1])  # Set cell color based on value
            cell.set_fontsize(10)

    # Adjust column widths
    table.auto_set_column_width(col=list(range(len(table_data.columns))))

    # Show the plot
    plt.show()

In [ ]:
# ======================
# Best subset regression with full metadata
# ======================
# This helper tests every possible combination of independent_variables for each dependent variable,
# then reports which variables were kept, which were removed, and the model-level evidence behind the decision.
#
# Main outputs:
# - best_models_summary: one-row-per-dependent-variable summary of the winning model
# - all_model_results: every model combination that was tested
# - variable_decision_metadata: one-row-per-dependent-variable/predictor inclusion-exclusion explanation
# - best_model_objects: fitted statsmodels model objects for the winning models

import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display


def _safe_model_metrics(model):
    """Collect common model metrics from a fitted statsmodels OLS model."""
    return {
        "r_squared": model.rsquared,
        "adjusted_r_squared": model.rsquared_adj,
        "aic": model.aic,
        "bic": model.bic,
        "f_statistic": model.fvalue,
        "f_pvalue": model.f_pvalue,
        "log_likelihood": model.llf,
        "residual_std_error": np.sqrt(model.mse_resid),
        "observations": int(model.nobs),
        "df_model": model.df_model,
        "df_residual": model.df_resid,
    }


def find_best_model_with_metadata(
    df,
    dependent_variables,
    independent_variables,
    selection_metric="adjusted_r_squared",
    pvalue_threshold=0.05,
    require_all_predictors_significant=False,
    min_predictors=1,
    max_predictors=None,
    verbose=True,
):
    """
    Brute-force best subset regression with full metadata.

    Parameters
    ----------
    df : pandas.DataFrame
        Source dataframe.
    dependent_variables : list[str] or str
        Dependent variable(s) to model.
    independent_variables : list[str]
        Candidate predictors to test in every possible combination.
    selection_metric : str
        Metric used to select the best model. Options: "adjusted_r_squared", "r_squared", "aic", "bic".
        For adjusted_r_squared/r_squared, larger is better. For aic/bic, smaller is better.
    pvalue_threshold : float
        Threshold used to describe whether kept variables are statistically significant.
    require_all_predictors_significant : bool
        If True, only models where every non-intercept predictor has p <= pvalue_threshold are eligible.
        If no models pass this filter, the function falls back to all tested models.
    min_predictors : int
        Smallest combination size to test.
    max_predictors : int or None
        Largest combination size to test. Defaults to all independent variables.
    verbose : bool
        If True, display the best-model summary, all-model results, and variable decision metadata.

    Returns
    -------
    best_models_summary : pandas.DataFrame
    all_model_results : pandas.DataFrame
    variable_decision_metadata : pandas.DataFrame
    best_model_objects : dict[str, statsmodels regression result]
    """
    if isinstance(dependent_variables, str):
        dependent_variables = [dependent_variables]

    valid_selection_metrics = {"adjusted_r_squared", "r_squared", "aic", "bic"}
    if selection_metric not in valid_selection_metrics:
        raise ValueError(f"selection_metric must be one of {sorted(valid_selection_metrics)}")

    if max_predictors is None:
        max_predictors = len(independent_variables)

    if min_predictors < 1:
        raise ValueError("min_predictors must be at least 1")
    if max_predictors > len(independent_variables):
        raise ValueError("max_predictors cannot exceed len(independent_variables)")
    if min_predictors > max_predictors:
        raise ValueError("min_predictors cannot be greater than max_predictors")

    larger_is_better = selection_metric in {"adjusted_r_squared", "r_squared"}

    all_rows = []
    best_rows = []
    decision_rows = []
    best_model_objects = {}

    for dependent_variable in dependent_variables:
        # Coerce relevant columns to numeric and drop rows that cannot be used by any tested model.
        model_data = (
            df[[dependent_variable] + list(independent_variables)]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if model_data.empty:
            raise ValueError(f"No usable rows remain for dependent variable '{dependent_variable}' after numeric coercion/dropna.")

        y = model_data[dependent_variable]

        for k in range(min_predictors, max_predictors + 1):
            for combo in itertools.combinations(independent_variables, k):
                X = sm.add_constant(model_data[list(combo)], has_constant="add")
                model = sm.OLS(y, X).fit()

                pvalues = model.pvalues.drop(labels=["const"], errors="ignore")
                coefficients = model.params.drop(labels=["const"], errors="ignore")
                significant_predictors = tuple(pvalues[pvalues <= pvalue_threshold].index)
                insignificant_predictors = tuple(pvalues[pvalues > pvalue_threshold].index)

                row = {
                    "dependent_variable": dependent_variable,
                    "variables": tuple(combo),
                    "variables_display": ", ".join(combo),
                    "num_predictors": len(combo),
                    "all_predictors_significant": bool((pvalues <= pvalue_threshold).all()),
                    "significant_predictors": significant_predictors,
                    "insignificant_predictors": insignificant_predictors,
                    "max_pvalue": pvalues.max() if len(pvalues) else np.nan,
                    "pvalues": pvalues.to_dict(),
                    "coefficients": coefficients.to_dict(),
                    "model": model,
                    **_safe_model_metrics(model),
                }
                all_rows.append(row)

        dep_results = pd.DataFrame([row for row in all_rows if row["dependent_variable"] == dependent_variable])

        eligible_results = dep_results.copy()
        if require_all_predictors_significant:
            eligible_results = eligible_results[eligible_results["all_predictors_significant"]]
            if eligible_results.empty:
                print(
                    f"No models for '{dependent_variable}' had all predictors significant at p <= {pvalue_threshold}. "
                    "Falling back to all tested models."
                )
                eligible_results = dep_results.copy()

        best_idx = (
            eligible_results[selection_metric].idxmax()
            if larger_is_better
            else eligible_results[selection_metric].idxmin()
        )
        best_row = dep_results.loc[best_idx].to_dict()
        best_model = best_row["model"]
        best_variables = tuple(best_row["variables"])
        best_model_objects[dependent_variable] = best_model

        best_rows.append({
            "dependent_variable": dependent_variable,
            "selection_metric": selection_metric,
            "best_variables_kept": best_variables,
            "best_variables_kept_display": ", ".join(best_variables),
            "variables_removed": tuple(v for v in independent_variables if v not in best_variables),
            "variables_removed_display": ", ".join(v for v in independent_variables if v not in best_variables),
            "num_predictors": best_row["num_predictors"],
            "r_squared": best_row["r_squared"],
            "adjusted_r_squared": best_row["adjusted_r_squared"],
            "aic": best_row["aic"],
            "bic": best_row["bic"],
            "f_pvalue": best_row["f_pvalue"],
            "observations": best_row["observations"],
            "all_predictors_significant": best_row["all_predictors_significant"],
            "max_pvalue": best_row["max_pvalue"],
            "reason_best_model_selected": (
                f"Selected because it had the {'highest' if larger_is_better else 'lowest'} "
                f"{selection_metric} among eligible tested combinations."
            ),
        })

        # Build variable-level keep/remove explanations.
        best_metric_value = best_row[selection_metric]
        for variable in independent_variables:
            with_var = dep_results[dep_results["variables"].apply(lambda vars_tuple: variable in vars_tuple)]
            without_var = dep_results[dep_results["variables"].apply(lambda vars_tuple: variable not in vars_tuple)]

            best_with = None
            best_without = None
            if not with_var.empty:
                best_with = with_var.loc[with_var[selection_metric].idxmax() if larger_is_better else with_var[selection_metric].idxmin()]
            if not without_var.empty:
                best_without = without_var.loc[without_var[selection_metric].idxmax() if larger_is_better else without_var[selection_metric].idxmin()]

            kept = variable in best_variables
            coefficient = best_model.params.get(variable, np.nan)
            pvalue = best_model.pvalues.get(variable, np.nan)

            if kept:
                if pd.notna(pvalue) and pvalue <= pvalue_threshold:
                    why = (
                        f"Kept: included in the best {selection_metric} model and statistically significant "
                        f"in that model (p={pvalue:.4g})."
                    )
                elif pd.notna(pvalue):
                    why = (
                        f"Kept: included in the best {selection_metric} model, but not statistically significant "
                        f"at p <= {pvalue_threshold} in that model (p={pvalue:.4g}). "
                        "It may still improve the selected model-level criterion."
                    )
                else:
                    why = f"Kept: included in the best {selection_metric} model."
            else:
                if best_with is not None:
                    metric_gap = (
                        best_metric_value - best_with[selection_metric]
                        if larger_is_better
                        else best_with[selection_metric] - best_metric_value
                    )
                    why = (
                        f"Removed: the best model containing this variable had a worse {selection_metric} "
                        f"than the selected model by {metric_gap:.6g}."
                    )
                else:
                    why = "Removed: this variable was not present in any eligible/tested model."

            decision_rows.append({
                "dependent_variable": dependent_variable,
                "variable": variable,
                "decision": "kept" if kept else "removed",
                "coefficient_in_best_model": coefficient,
                "pvalue_in_best_model": pvalue,
                "significant_in_best_model": bool(pd.notna(pvalue) and pvalue <= pvalue_threshold),
                f"best_{selection_metric}_with_variable": np.nan if best_with is None else best_with[selection_metric],
                f"best_{selection_metric}_without_variable": np.nan if best_without is None else best_without[selection_metric],
                f"avg_{selection_metric}_with_variable": np.nan if with_var.empty else with_var[selection_metric].mean(),
                f"avg_{selection_metric}_without_variable": np.nan if without_var.empty else without_var[selection_metric].mean(),
                "why": why,
            })

    all_model_results = pd.DataFrame(all_rows).sort_values(
        ["dependent_variable", selection_metric],
        ascending=[True, not larger_is_better],
    ).reset_index(drop=True)

    # Keep the model object available in best_model_objects, but remove it from display-oriented summary tables.
    all_model_results_for_display = all_model_results.drop(columns=["model"], errors="ignore")
    best_models_summary = pd.DataFrame(best_rows)
    variable_decision_metadata = pd.DataFrame(decision_rows)

    if verbose:
        print("Best model summary")
        display(best_models_summary)

        print("Variable keep/remove metadata")
        display(variable_decision_metadata)

        print("All tested model combinations, sorted by selected metric")
        display(all_model_results_for_display)

        for dependent_variable, model in best_model_objects.items():
            print(f"\nRegression summary for best model: {dependent_variable}")
            print(model.summary())

    return best_models_summary, all_model_results, variable_decision_metadata, best_model_objects


# ======================
# Example usage
# ======================
# After defining df, dependent_variables, and independent_variables, run:
#
# best_models_summary, all_model_results, variable_decision_metadata, best_model_objects = find_best_model_with_metadata(
#     df=df,
#     dependent_variables=dependent_variables,
#     independent_variables=independent_variables,
#     selection_metric="adjusted_r_squared",  # Options: "adjusted_r_squared", "r_squared", "aic", "bic"
#     pvalue_threshold=0.05,
#     require_all_predictors_significant=False,
#     verbose=True,
# )
#
# To export the metadata:
# best_models_summary.to_csv("best_models_summary.csv", index=False)
# all_model_results.drop(columns=["model"], errors="ignore").to_csv("all_model_results.csv", index=False)
# variable_decision_metadata.to_csv("variable_decision_metadata.csv", index=False)
